# Topic 15 — Hierarchical Clustering

> **Goal of this notebook:** understand agglomerative (bottom-up) clustering — repeatedly merging the closest groups into a tree (**dendrogram**) — the different **linkage** rules, and how to cut the tree into clusters.

**Contents**
1. Concept & intuition
2. The mechanics (linkage criteria, distances)
3. Reading a dendrogram & choosing clusters
4. The data: synthetic blobs
5. Building & plotting a dendrogram
6. Cutting the tree into clusters
7. Comparing linkage methods
8. Pros, cons & when to use
9. Summary

## 1. Concept & Intuition

Hierarchical clustering builds a **hierarchy** of clusters instead of a single flat partition. The common **agglomerative** (bottom-up) form:
1. start with every point as its own cluster,
2. repeatedly **merge the two closest clusters**,
3. continue until one cluster remains.

The full sequence of merges is drawn as a **dendrogram** — a tree whose branch heights show the distance at which clusters joined. Unlike K-Means you **don't pick `k` up front**; you cut the tree afterwards at whatever level you like. (The opposite, top-down **divisive** approach splits one big cluster repeatedly.)

## 2. The Mechanics: Linkage Criteria

"Closest" depends on how we measure distance **between clusters** (the *linkage*):

| Linkage | Distance between clusters A, B | Tendency |
|---------|-------------------------------|----------|
| **Single** | nearest pair of points | long, chained clusters |
| **Complete** | farthest pair of points | compact, equal-diameter clusters |
| **Average** | average of all pairwise distances | balance of the two |
| **Ward** | increase in total within-cluster variance | compact, similar-sized (most popular) |

The point-to-point distance is usually **Euclidean** (Ward requires it).

## 3. Reading a Dendrogram & Choosing Clusters

- The **height** of a merge = how dissimilar the joined clusters are. Short merges = similar groups; tall jumps = dissimilar.
- To get `k` clusters, draw a horizontal line that crosses exactly `k` vertical branches — equivalently, cut just below the **largest vertical gap**.
- No need to refit for different `k`; just cut at a different height.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster

from sklearn.datasets import make_blobs
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import adjusted_rand_score

np.random.seed(42)
plt.rcParams['figure.figsize'] = (8, 5)
print('Libraries loaded.')

## 4. The Data: Synthetic Blobs

In [ ]:
X, y_true = make_blobs(n_samples=150, centers=3, cluster_std=1.0, random_state=42)
plt.scatter(X[:, 0], X[:, 1], s=15, color='gray')
plt.title('Raw data (3 true blobs)'); plt.show()

## 5. Building & Plotting a Dendrogram

We use **Ward** linkage. The big vertical gap near the top reveals the natural number of clusters.

In [ ]:
Z = linkage(X, method='ward')
plt.figure(figsize=(12, 5))
dendrogram(Z, truncate_mode='lastp', p=20, leaf_rotation=90, show_contracted=True)
plt.title('Dendrogram (Ward linkage)')
plt.xlabel('samples (truncated)'); plt.ylabel('merge distance')
plt.show()

## 6. Cutting the Tree into Clusters

`fcluster` cuts the dendrogram; here we ask for 3 clusters and check agreement with the true blobs.

In [ ]:
labels = fcluster(Z, t=3, criterion='maxclust')
print('Adjusted Rand Index vs true blobs:', round(adjusted_rand_score(y_true, labels), 3))

plt.scatter(X[:, 0], X[:, 1], c=labels, cmap='viridis', s=15)
plt.title('Clusters from cutting the dendrogram (k=3)'); plt.show()

## 7. Comparing Linkage Methods

Different linkages can give very different clusterings on the same data.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, method in zip(axes, ['single', 'complete', 'average', 'ward']):
    lab = AgglomerativeClustering(n_clusters=3, linkage=method).fit_predict(X)
    ari = adjusted_rand_score(y_true, lab)
    ax.scatter(X[:, 0], X[:, 1], c=lab, cmap='viridis', s=12)
    ax.set_title(f'{method}\nARI={ari:.2f}')
plt.show()

## 8. Pros, Cons & When to Use

**Pros**
- **No need to pre-specify `k`** — cut the dendrogram afterwards.
- The dendrogram is a rich, interpretable visualisation of nested structure.
- Works with any distance metric (most linkages).

**Cons**
- **Slow**: O(n²) memory / O(n³) time — impractical for very large datasets.
- Merges are **greedy and final** — a bad early merge can't be undone.
- Sensitive to noise and the choice of linkage (single-linkage chains).

**When to use**
- Small/medium datasets where you want to explore structure at multiple granularities.
- Taxonomy/phylogeny-style problems and when a dendrogram aids interpretation.

## 9. Summary

- Agglomerative clustering repeatedly **merges the closest clusters**, recording the process as a **dendrogram**.
- The **linkage** rule (single/complete/average/**ward**) defines inter-cluster distance and strongly shapes results.
- You choose the number of clusters **after** fitting by cutting the tree — here k=3 recovered the blobs (high ARI), with Ward and complete linkage performing best.